<a href="https://colab.research.google.com/github/sneha-galani/msba_supplychain/blob/main/location_of_a_parcel_pick_up_facility.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Location of a parcel pick up facility (top-10 neighborhoods by population)


In [11]:
import math
import numpy as np
import pandas as pd
import requests
import time
from urllib.parse import quote
from IPython.display import display

In [12]:
import pandas as pd
import requests
from io import StringIO

# Wikipedia page with Atlanta neighborhood populations
WIKI_URL = "https://en.wikipedia.org/wiki/Table_of_Atlanta_neighborhoods_by_population"

# Request the page using a browser-like user agent (avoids 403 error)
headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(WIKI_URL, headers=headers)
response.raise_for_status()

# Read all tables from the page HTML
tables = pd.read_html(StringIO(response.text))

# Select the main population table (largest table on the page)
tbl = max(tables, key=lambda t: t.shape[0])

# Clean column names
tbl.columns = [c.strip() for c in tbl.columns]

# Find the neighborhood and population columns
name_col = [c for c in tbl.columns if "Neighborhood" in c][0]
pop_col = [c for c in tbl.columns if "Population" in c][0]

# Keep only the needed columns
df = tbl[[name_col, pop_col]].copy()
df.columns = ["neighborhood", "population"]

# Convert population to integer
df["population"] = (
    df["population"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.extract(r"(\d+)")
    .astype(int)
)

# Select the 10 most populous neighborhoods
top10 = (
    df.sort_values("population", ascending=False)
      .head(10)
      .reset_index(drop=True)
)

top10

,neighborhood,population
0,Midtown,16569
1,Downtown,13411
2,Old Fourth Ward,10505
3,North Buckhead,8270
4,Pine Hills,8033
5,Morningside/Lenox Park,8030
6,Virginia-Highland,7800
7,Grant Park,6771
8,Georgia Tech,6607
9,Kirkwood,5897


In [13]:
# --- Get coordinates for each neighborhood ---
# Try OpenStreetMap (Nominatim) first.
# If it fails or is rate-limited, use predefined coordinates.

import time

MANUAL_COORDS = {
    "Midtown": (33.7868014, -84.3795169),
    "Downtown": (33.75500, -84.39000),
    "Old Fourth Ward": (33.766, -84.372),
    "North Buckhead": (33.845556, -84.378333),
    "Pine Hills": (33.83583, -84.36389),
    "Morningside/Lenox Park": (33.797519, -84.358295),
    "Virginia-Highland": (33.7824, -84.3543),
    "Grant Park": (33.736, -84.371),
    "Georgia Tech": (33.776, -84.396),
    "Kirkwood": (33.752095, -84.323668),
}

def nominatim_search(query):
    """Return (lat, lon) for a location using OpenStreetMap."""
    url = "https://nominatim.openstreetmap.org/search"
    params = {"format": "jsonv2", "q": query, "limit": 1}
    headers = {"User-Agent": "atl-pickup-location-class-project"}

    r = requests.get(url, params=params, headers=headers)
    r.raise_for_status()

    results = r.json()
    if not results:
        return None

    return float(results[0]["lat"]), float(results[0]["lon"])

# Look up coordinates for each neighborhood
lats, lons, sources = [], [], []

for name in top10["neighborhood"]:
    query = f"{name}, Atlanta, GA"

    try:
        coords = nominatim_search(query)
    except Exception:
        coords = None

    # Pause briefly to avoid Nominatim rate limits
    time.sleep(1)

    if coords is None:
        if name in MANUAL_COORDS:
            coords = MANUAL_COORDS[name]
            sources.append("manual")
        else:
            coords = (np.nan, np.nan)
            sources.append("missing")
    else:
        sources.append("nominatim")

    lats.append(coords[0])
    lons.append(coords[1])

# Add coordinates and source to the dataframe
top10["lat"] = lats
top10["lon"] = lons
top10["coord_source"] = sources

top10

,neighborhood,population,lat,lon,coord_source
0,Midtown,16569,33.781656,-84.384071,nominatim
1,Downtown,13411,33.763820,-84.385607,nominatim
2,Old Fourth Ward,10505,33.764108,-84.371763,nominatim
3,North Buckhead,8270,33.877277,-84.366380,nominatim
4,Pine Hills,8033,33.563652,-84.259535,nominatim
5,Morningside/Lenox Park,8030,33.805430,-84.356930,nominatim
6,Virginia-Highland,7800,33.782656,-84.353691,nominatim
7,Grant Park,6771,33.735862,-84.370932,nominatim
8,Georgia Tech,6607,33.776095,-84.398808,nominatim
9,Kirkwood,5897,33.752427,-84.323154,nominatim


In [14]:
# --- Haversine distance (great-circle) ---
R_KM = 6371.0  # Earth radius in kilometers

def haversine_km(lat1, lon1, lat2, lon2):
    """Return straight-line distance (km) between two lat/lon points."""
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlmb = math.radians(lon2 - lon1)

    a = (math.sin(dphi / 2) ** 2 +
         math.cos(phi1) * math.cos(phi2) * math.sin(dlmb / 2) ** 2)

    return 2 * R_KM * math.asin(math.sqrt(a))

# --- Find the best pickup point using straight-line distance ---
# This uses a weighted geometric median (weights = population).
# We convert lat/lon to a simple local (x, y) grid so Weiszfeld's method works.

def weighted_geometric_median_local(lat, lon, w, max_iter=5000, tol=1e-9):
    """Return (lat, lon) that minimizes weighted straight-line distance."""

    # Reference latitude for local projection
    lat0 = math.radians(np.average(lat, weights=w))

    # Convert lat/lon to x,y in km (local projection)
    x = np.radians(lon) * math.cos(lat0) * R_KM
    y = np.radians(lat) * R_KM

    # Start at the weighted average point
    x0 = np.average(x, weights=w)
    y0 = np.average(y, weights=w)

    eps = 1e-12

    # Weiszfeld algorithm loop
    for _ in range(max_iter):
        d = np.hypot(x0 - x, y0 - y)

        # If we land right on a neighborhood point, stop
        if np.any(d < eps):
            j = int(np.argmin(d))
            x0, y0 = x[j], y[j]
            break

        # Update step
        x1 = np.sum(w * x / d) / np.sum(w / d)
        y1 = np.sum(w * y / d) / np.sum(w / d)

        # Stop if the update is tiny
        if math.hypot(x1 - x0, y1 - y0) < tol:
            x0, y0 = x1, y1
            break

        x0, y0 = x1, y1

    # Convert x,y back to lat/lon
    lat_med = math.degrees(y0 / R_KM)
    lon_med = math.degrees(x0 / (R_KM * math.cos(lat0)))

    return lat_med, lon_med

# Inputs: neighborhood coordinates and population weights
lat = top10["lat"].to_numpy()
lon = top10["lon"].to_numpy()
w = top10["population"].to_numpy().astype(float)

# Optimal pickup location using haversine/straight-line distance
haversine_opt = weighted_geometric_median_local(lat, lon, w)
haversine_opt

(33.771420236763724, -84.37613810844012)

In [15]:
# --- Driving distance / time using OSRM ---
# We use the OSRM public demo server (OpenStreetMap routing).
# This returns driving travel times (in seconds) between points.

OSRM_TABLE = "https://router.project-osrm.org/table/v1/driving/"

def osrm_table_durations(coords_lonlat):
    """
    Return a matrix of driving times (seconds) between all points.
    coords_lonlat should be a list of (lon, lat).
    """
    coord_str = ";".join([f"{lon},{lat}" for lon, lat in coords_lonlat])
    url = OSRM_TABLE + coord_str
    params = {"annotations": "duration"}

    r = requests.get(url, params=params)
    r.raise_for_status()

    return r.json()

# Coordinates in (lon, lat) format as required by OSRM
coords_lonlat = list(zip(top10["lon"], top10["lat"]))

# Call OSRM to get the travel-time matrix
osrm_json = osrm_table_durations(coords_lonlat)

# Extract driving durations (seconds)
dur = np.array(osrm_json["durations"])

dur.shape

(10, 10)

In [16]:
# --- Choose best pickup location using driving time ---
# We want to minimize: sum_i (population_i * time(neighborhood_i -> facility))

w = top10["population"].to_numpy().astype(float)

# dur[i, j] = time from neighborhood i -> candidate facility j
weighted_cost_to_facility = (dur.T * w).sum(axis=1)

best_idx = int(np.argmin(weighted_cost_to_facility))

drive_opt = (
    float(top10.loc[best_idx, "lat"]),
    float(top10.loc[best_idx, "lon"]),
    top10.loc[best_idx, "neighborhood"],
)

drive_opt

(33.7641085, -84.3717629, 'Old Fourth Ward')

In [17]:
# --- Convert coordinates to street addresses ---
# Use OpenStreetMap (Nominatim) reverse geocoding

def nominatim_reverse(lat, lon):
    """Return a readable address for a latitude/longitude pair."""
    url = "https://nominatim.openstreetmap.org/reverse"
    params = {"format": "jsonv2", "lat": lat, "lon": lon}
    headers = {"User-Agent": "atl-pickup-location-class-project"}

    r = requests.get(url, params=params, headers=headers)
    r.raise_for_status()

    data = r.json()
    return data.get("display_name")

# Address for haversine-optimal location
hav_addr = nominatim_reverse(haversine_opt[0], haversine_opt[1])

# Address for driving-time-optimal location
drv_addr = nominatim_reverse(drive_opt[0], drive_opt[1])

hav_addr, drv_addr

('355, North Avenue Northeast, Old Fourth Ward, Atlanta, Fulton County, Georgia, 30308, United States',
 'Ralph McGill Boulevard, Inman Park, Old Fourth Ward, Atlanta, Fulton County, Georgia, 30308, United States')

In [18]:
import folium

# Create base map centered on the neighborhoods
m = folium.Map(
    location=[top10["lat"].mean(), top10["lon"].mean()],
    zoom_start=12
)

# Add neighborhood markers
for _, r in top10.iterrows():
    folium.CircleMarker(
        location=[r["lat"], r["lon"]],
        radius=4,
        popup=f'{r["neighborhood"]} (pop {r["population"]:,})',
        tooltip=r["neighborhood"],
    ).add_to(m)

# Add haversine-optimal facility location
folium.Marker(
    location=[haversine_opt[0], haversine_opt[1]],
    popup=f"Haversine optimum\n{hav_addr}",
    tooltip="Haversine optimum",
    icon=folium.Icon(color="green")
).add_to(m)

# Add driving-time-optimal facility location
folium.Marker(
    location=[drive_opt[0], drive_opt[1]],
    popup=f"Driving optimum (candidate: {drive_opt[2]})\n{drv_addr}",
    tooltip="Driving optimum",
    icon=folium.Icon(color="blue")
).add_to(m)

display(m)

In [19]:
print("Top-10 neighborhoods used:")
display(top10[["neighborhood","population","lat","lon","coord_source"]])

print("\nHaversine-optimal facility coordinate:", haversine_opt)
print("Haversine-optimal address:", hav_addr)

print("\nDriving-optimal facility (candidate centroid):", drive_opt)
print("Driving-optimal address:", drv_addr)

m.save("atl_pickup_facility_map.html")
print("\nSaved map to atl_pickup_facility_map.html")

Top-10 neighborhoods used:


,neighborhood,population,lat,lon,coord_source
0,Midtown,16569,33.781656,-84.384071,nominatim
1,Downtown,13411,33.763820,-84.385607,nominatim
2,Old Fourth Ward,10505,33.764108,-84.371763,nominatim
3,North Buckhead,8270,33.877277,-84.366380,nominatim
4,Pine Hills,8033,33.563652,-84.259535,nominatim
5,Morningside/Lenox Park,8030,33.805430,-84.356930,nominatim
6,Virginia-Highland,7800,33.782656,-84.353691,nominatim
7,Grant Park,6771,33.735862,-84.370932,nominatim
8,Georgia Tech,6607,33.776095,-84.398808,nominatim
9,Kirkwood,5897,33.752427,-84.323154,nominatim



Haversine-optimal facility coordinate: (33.771420236763724, -84.37613810844012)
Haversine-optimal address: 355, North Avenue Northeast, Old Fourth Ward, Atlanta, Fulton County, Georgia, 30308, United States

Driving-optimal facility (candidate centroid): (33.7641085, -84.3717629, 'Old Fourth Ward')
Driving-optimal address: Ralph McGill Boulevard, Inman Park, Old Fourth Ward, Atlanta, Fulton County, Georgia, 30308, United States

Saved map to atl_pickup_facility_map.html


(i) Facility locations

Using the 10 most populous Atlanta neighborhoods from Wikipedia, we identified two parcel pick-up facility locations that minimize one-way travel, weighted by neighborhood population.

Haversine (straight-line distance) optimal location:
355 North Avenue NE, Old Fourth Ward, Atlanta, GA 30308, United States
(Coordinates: 33.77142, −84.37614)

Driving-distance optimal location (OSRM routing):
Ralph McGill Boulevard, Old Fourth Ward / Inman Park, Atlanta, GA 30308, United States
(Coordinates: 33.76411, −84.37176)

The two locations are geographically close but not identical. The haversine-optimal point lies slightly north, reflecting purely geometric centrality, while the driving-optimal point shifts slightly southeast, reflecting road-network connectivity and driving times.

Geocoding and routing were performed using OpenStreetMap services (Nominatim and OSRM), which provide open, reproducible alternatives to commercial mapping APIs and avoid the need for API keys or billing.